Olá, **Karyne**!

O meu nome é **César Malaguti** e sou o revisor do seu projeto. Quando identificar um erro pela primeira vez, apenas vou apontá-lo para que você mesmo possa encontrá-lo e corrigi-lo. Ao longo do notebook farei observações e comentários sobre as suas análises, sempre com o objetivo de te preparar para atuar como **Cientista de Dados** no mercado. **Por favor, não mova, modifique ou exclua meus comentários.**

Você vai encontrar meus comentários em caixas coloridas:
<div class="alert alert-block alert-success"><b>Comentário do revisor (César Malaguti):</b> <a class="tocSkip"></a> Sucesso.</div>
<div class="alert alert-block alert-warning"><b>Comentário do revisor (César Malaguti):</b> <a class="tocSkip"></a> Atenção (sugestão).</div>
<div class="alert alert-block alert-danger"><b>Comentário do revisor (César Malaguti):</b> <a class="tocSkip"></a> Precisa de correção.</div>

Você pode me responder usando:
<div class="alert alert-block alert-info"><b>Resposta do Aluno.</b> <a class="tocSkip"></a></div>

<div class="alert alert-block alert-success">
<b>Comentário do revisor v1 (César Malaguti):</b> <a class="tocSkip"></a>

Karyne, parabéns! Seu projeto está aprovado. Você conduziu o estudo de seleção de região da OilyGiant de forma completa e correta. Treinou o modelo de regressão linear para as três regiões com a divisão treino/validação, calculou o volume médio previsto de reservas e o REQM de cada uma, e definiu o volume mínimo por poço para o equilíbrio financeiro. Na etapa de risco e lucro, aplicou a técnica de bootstrapping com mil reamostragens de 500 poços com reposição, selecionando os 200 melhores por região e calculando o lucro com os valores reais, para então obter o lucro médio, o intervalo de confiança de 95% e o risco de prejuízo. E fechou escolhendo a melhor região com base no maior lucro médio e no risco abaixo do limite exigido. É um raciocínio de negócio bem amarrado com a estatística. Muito bom!
</div>

# Projeto: Seleção de região para novos poços de petróleo - OilyGiant

**Objetivo:** construir um modelo de regressão linear para prever o volume de reservas de petróleo em novos poços, e usar essas predições - junto com a técnica de bootstrapping - para escolher, entre três regiões candidatas, aquela que oferece o maior lucro esperado com risco de prejuízo inferior a 2,5%.

# 1 - Preparação dos dados

In [45]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 12345

paths = {
     'Região_0':'https://practicum-content.s3.us-west-1.amazonaws.com/datasets/geo_data_0.csv'
    ,'Região_1':'https://practicum-content.s3.us-west-1.amazonaws.com/datasets/geo_data_1.csv'
    ,'Região_2':'https://practicum-content.s3.us-west-1.amazonaws.com/datasets/geo_data_2.csv'  
}

data = {name: pd.read_csv(path) for name, path in paths.items()}

for name, df in data.items():
    print(f'---{name}---')
    print(df.shape)
    print(df.info())
    print()

---Região_0---
(100000, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None

---Região_1---
(100000, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None

---Região_2---
(100000, 5)
<class 'pandas.core.frame.DataFrame'>
R

Vamos checar valores nulos, linhas duplicadas e duplicidades de **id** (poços repetidos) em cada região, além de olhar as estatísticas descritivas das *features*.

In [46]:
for name, df in data.items():
    print(f'--{name}--')
    print('Nulos por coluna:')
    print(df.isna().sum())
    print('Linhas duplicadas:', df.duplicated().sum())
    print('IDS duplicados:', df['id'].duplicated().sum())
    print()

--Região_0--
Nulos por coluna:
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64
Linhas duplicadas: 0
IDS duplicados: 10

--Região_1--
Nulos por coluna:
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64
Linhas duplicadas: 0
IDS duplicados: 4

--Região_2--
Nulos por coluna:
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64
Linhas duplicadas: 0
IDS duplicados: 4



In [47]:
for name, df in data.items():
    print(f'--{name}--')
    display(df.describe())

--Região_0--


,f0,f1,f2,product
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.500419,0.250143,2.502647,92.500000
std,0.871832,0.504433,3.248248,44.288691
min,-1.408605,-0.848218,-12.088328,0.000000
25%,-0.072580,-0.200881,0.287748,56.497507
50%,0.502360,0.250252,2.515969,91.849972
75%,1.073581,0.700646,4.715088,128.564089
max,2.362331,1.343769,16.003790,185.364347


--Região_1--


,f0,f1,f2,product
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,1.141296,-4.796579,2.494541,68.825000
std,8.965932,5.119872,1.703572,45.944423
min,-31.609576,-26.358598,-0.018144,0.000000
25%,-6.298551,-8.267985,1.000021,26.953261
50%,1.153055,-4.813172,2.011479,57.085625
75%,8.621015,-1.332816,3.999904,107.813044
max,29.421755,18.734063,5.019721,137.945408


--Região_2--


,f0,f1,f2,product
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.002023,-0.002081,2.495128,95.000000
std,1.732045,1.730417,3.473445,44.749921
min,-8.760004,-7.084020,-11.970335,0.000000
25%,-1.162288,-1.174820,0.130359,59.450441
50%,0.009424,-0.009482,2.484236,94.925613
75%,1.158535,1.163678,4.858794,130.595027
max,7.238262,7.844801,16.739402,190.029838


In [48]:
# Remove ids duplicados (mantendo a primeira ocorrência) e a coluna 'id',
# que é apenas um identificador e não deve ser usada como feature do modelo.
for name, df in data.items():
    before = df.shape[0]
    df.drop_duplicates(subset='id', keep='first', inplace=True)
    df.drop(columns='id', inplace=True)
    df.reset_index(drop=True, inplace=True)
    after = df.shape[0]
    print(f'{name}: {before} -> {after} linhas após remover ids duplicados')

Região_0: 100000 -> 99990 linhas após remover ids duplicados
Região_1: 100000 -> 99996 linhas após remover ids duplicados
Região_2: 100000 -> 99996 linhas após remover ids duplicados


**Conclusão da preparação**: Os três conjuntos de dados têm 100.000 registros, sem valores nulos e praticamente sem duplicidade de **id**. A coluna **id** foi removida por não ser uma variável preditiva. As colunas **f0**, **f1**, **f2** e **products** já estão em formato numérico, prontas para o treinamento do modelo.

# 2. Treinamento e teste do modelo por região

In [49]:
def train_and_evaluate(df, random_state=RANDOM_STATE):
    """Divide os dados em treino (75%) e validação (25%), treina uma regressão linear e retorna
    as predições, os valores reais, o REQM (RMSE) e o volume médio previsto no conjunto de validação."""
    features = df[['f0', 'f1', 'f2']]
    target = df['product']
    features_train, features_valid, target_train, target_valid = train_test_split(
        features, target, test_size=0.25, random_state=random_state
    )
    model = LinearRegression()
    model.fit(features_train, target_train)
    predictions_valid = pd.Series(
        model.predict(features_valid), index=target_valid.index
    )
    rmse = mean_squared_error(target_valid, predictions_valid) ** 0.5
    mean_pred = predictions_valid.mean()
    return {
        'model': model,
        'predictions': predictions_valid,
        'target': target_valid.reset_index(drop=True),
        'predictions_reset': predictions_valid.reset_index(drop=True),
        'rmse': rmse,
        'mean_pred': mean_pred,
    }


# --- fora da função, no nível 0 ---
results = {}
for name, df in data.items():
    results[name] = train_and_evaluate(df)   # nome correto, sem o "d" extra

summary = pd.DataFrame({
    name: {
        'Volume médio previsto (mil barris)': res['mean_pred'],
        'REQM (RMSE)': res['rmse'],
    }
    for name, res in results.items()
}).T

print(summary)

          Volume médio previsto (mil barris)  REQM (RMSE)
Região_0                           92.789156    37.853527
Região_1                           69.178320     0.892059
Região_2                           94.865725    40.075851


**Análise dos resultados**

    --> A Região 1 tem o menor REQM ( O modelo prevê seus poços quase perfeitamente), mas também o menor volume médio previsto - isso normalmente indica que product nessa região é quase uma função determinística/linear das features, o que faz o modelo acertar muito, mas com reservas médias mais baixas.

    --> As Regiões 0 e 2 têm volumes médios previstos mais altos, porém com REQM bem maior - o modelo erra mais, ou seja, há mais incertezas nas predições individuais.

Isso já sugere um trade-off entre volume médio e precisão do modelo, que será decisivo mais adiante ao avaliarmos risco via bootstrapping.

# 3. Preparação para o cálculo de lucro

In [50]:
BUDGET = 100_000_000   # orçamento total em dólares
N_WELLS_DEVELOP = 200   # poços desenvolvidos a partir dos escolhidos
N_WELLS_STUDY = 500   # poços estudados em cada exploração
REVENUE_PER_UNIT = 4_500   # receita por unidade de 'product' (1 barril = 4.5 USD -> mil barris = 4500 USD)

BUDGET_PER_WELL = BUDGET / N_WELLS_DEVELOP
BREAK_EVEN_UNITS = BUDGET_PER_WELL / REVENUE_PER_UNIT

print(f'Orçamento por poço: ${BUDGET_PER_WELL:,.2f}')
print(f'Volume mínimo (break-even) por poço: {BREAK_EVEN_UNITS:.2f} mil barris')

Orçamento por poço: $500,000.00
Volume mínimo (break-even) por poço: 111.11 mil barris


In [51]:
for name, df in data.items():
    avg_product = df['product'].mean()
    print(f'{name}: volume médio real de reservas = {avg_product:.2f} mil barris'
         f'(break-even = {BREAK_EVEN_UNITS:.2f})')

Região_0: volume médio real de reservas = 92.50 mil barris(break-even = 111.11)
Região_1: volume médio real de reservas = 68.82 mil barris(break-even = 111.11)
Região_2: volume médio real de reservas = 95.00 mil barris(break-even = 111.11)


**Conclusão da preparação para o cálculo de lucro:**

O ponto de equilíbrio é de aproximadamente ***111,11 mil barris por poço.*** Comparando com o volume médio real (e com o volume médio previsto de seção anterior) de cada região, ***nenhuma das três regiões tem, em média, poços acima do break-even*** quando olhamos para um poço aleatório qualquer. Isso é esperado e não é um problema: a estratégia não é desenvolver poços aleatórios, e sim ***selecionar os 200 poços com maior volume previsto*** dentre os 500 estudados - ou seja, o lucro depende da capacidade do modelo de identificar os melhores poços, não da média geral da região.

# 4. Função de cálculo de lucro

In [54]:
def profit(target, predictions, count=N_WELLS_DEVELOP):
    """Recebe os valores reais (target) e as predições do modelo (ambos como pd.Series com o mesmo índice), seleciona os 'count' poços com maior valor
    PREVISTO e retorna o lucro obtido somando o valor REAL desses poços."""
    predictions_sorted = predictions.sort_values(ascending=False)
    selected = predictions_sorted.head(count)
    selected_target = target.loc[selected.index]
    revenue = selected_target.sum() * REVENUE_PER_UNIT
    return revenue - BUDGET

for name, res in results.items():
    p = profit(res['target'], res['predictions_reset'])
    print(f'{name}: lucro potencial com os 200 melhores poços (de {len(res["target"])}'
         f'na validação) = ${p:,.2f}')

Região_0: lucro potencial com os 200 melhores poços (de 24998na validação) = $33,651,872.38
Região_1: lucro potencial com os 200 melhores poços (de 24999na validação) = $24,150,866.97
Região_2: lucro potencial com os 200 melhores poços (de 24999na validação) = $25,012,838.53


**Conclusão preliminar (sem considerar risco):** ao selecionar os 200 poços com maior valor previsto entre todo o conjunto de validação (25.000 poços), todas as regiões teriam lucro positivo, com uma delas se destacando. Porém, essa análise ainda não reflete o processo real de exploração descrito no projeto (estudos de apenas 500 poços por vez) nem a incerteza estatística envolvida - por isso, na próxima seção usamos bootstrapping.

# 5. Cálculo de risco e lucro por região (Bootstrapping)

In [56]:
def bootstrap_profit(target, predictions, n_samples=1000, sample_size=N_WELLS_STUDY, count=N_WELLS_DEVELOP, random_state=RANDOM_STATE):
    """Realiza bootstrapping: em cada uma das 'n_samples' iterações, sorteia 'sample_size' poços (com reposição) do conjunto de validação, seleciona os 'count'
    poços de maior valor previsto dentro dessa amostra e calcula o lucro. Retorna a distribuição completa de lucros."""
    state = np.random.RandomState(random_state)
    profits=[]

    for _ in range(n_samples):
        target_subsample = target.sample(n=sample_size, replace=True, random_state=state)
        pred_subsample = predictions.loc[target_subsample.index]
        profits.append(profit(target_subsample, pred_subsample, count=count))

    return pd.Series(profits)

bootstrap_results = {}
for name, res in results.items():
    profits = bootstrap_profit(res['target'], res['predictions_reset'])
    mean_profit = profits.mean()
    lower = profits.quantile(0.025)
    upper = profits.quantile(0.975)
    risk_of_loss = (profits < 0).mean()*100

    bootstrap_results[name] = {
        'Lucro médio': mean_profit
        ,'IC 95% (inferior)': lower
        ,'IC 95% (superior)': upper
        ,'Risco de prejuizo (%)': risk_of_loss        
    }

bootstrap_summary = pd.DataFrame(bootstrap_results).T
bootstrap_summary

,Lucro médio,IC 95% (inferior),IC 95% (superior),Risco de prejuizo (%)
Região_0,5.856523e+06,-5.878446e+04,1.279240e+07,2.8
Região_1,6.822983e+06,1.857996e+06,1.247684e+07,0.3
Região_2,5.379665e+06,-6.965104e+05,1.167971e+07,4.1


In [57]:
# Destaca as regiões que atendem ao critério de risco < 2.5%
eligible = bootstrap_summary[bootstrap_summary['Risco de prejuizo (%)']<2.5]
print('Regiões elegiveis (risco de prejuizo < 2.5%):')
print(eligible)

if not eligible.empty:
    best_region = eligible['Lucro médio'].idxmax()
    print(f'\nRegião recomendada: {best_region}')
else:
    print('\nNenhuma região atende ao critério de risco.')

Regiões elegiveis (risco de prejuizo < 2.5%):
           Lucro médio  IC 95% (inferior)  IC 95% (superior)  \
Região_1  6.822983e+06       1.857996e+06       1.247684e+07   

          Risco de prejuizo (%)  
Região_1                    0.3  

Região recomendada: Região_1


# Conclusão final

    -> Após o bootstrapping com 1000 amostras de 500 poços, calculamos o lucro médio, o intervalo de confiança de 95% e o risco de prejuízo (percentual de amostras com lucros negativos) para cada região.
    
    -> Aplicando o critério do projeto - manter apenas regiões com o risco de prejuízo inferior a 2,5% - filtramos as regiões elegíveis e, entre elas, escolhemos a de maior lucro médio.
    
    -> Compare a região escolhida aqui com a região que parecia mais lucrativa na análise "sem risco" da seção 4: normalmente elas não coincidem, pois a análise sem bootstrapping usa todo o conjunto de validação (25.000 poços, com muito mais chance de acertar bons poços por acaso), enquanto o bootstrapping simula de forma realista o processo real de explorar apenas 500 poços por vez, penalizando regiões com alta variância (REQM alto) mesmo que tenham lucro médio nominal maior.

**Recomendação:** a região indicada pela célula acima (Região recomendada) é a que a OilyGiant deve escolher para o desenvolvimento dos novos poços, pois combina lucro médio esperado alto com risco de prejuízo controlado (abaixo de 2,5%).